# Phase-3 TRAIN targets

Use a fresh T4 runtime. Run the two code cells in order. Resume with the same Drive folder after a disconnect. Outputs stay sealed in the ZIP; no response previews.

Return only `human_check.csv` after a bilingual human fills every flagged row with pass/fail/uncertain and a reason. Q is a provisional lower bound until review; Q <120 after review halts. Freeze is a separate CPU command after the return; this notebook never trains.


In [ ]:
BLOB_TEXT = '...'
BLOB_SHA256 = 'd174fe0488b76f1508e58049033a6c83b1c9e382bda33bdc4fc9465313a053af'
# Unpack the frozen TRAIN-only payload and authenticate every runtime file.
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path
PACKAGE = Path('/content/phase3_targets') if Path('/content').exists() else Path('/tmp/phase3_targets')
PACKAGE.mkdir(parents=True, exist_ok=True)
BLOB = base64.b64decode(BLOB_TEXT)
assert hashlib.sha256(BLOB).hexdigest() == BLOB_SHA256
with zipfile.ZipFile(io.BytesIO(BLOB)) as archive:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    assert not any('dev.v2.json' in n.lower() or 'final_test' in n.lower() for n in archive.namelist())
    archive.extractall(PACKAGE)
sys.path.insert(0, str(PACKAGE))
from t4_smoke import phase3_targets as pt
contract, _, _ = pt.verify_contract(PACKAGE)
# torch must be the Colab build; never install or replace it.
import importlib.metadata
assert importlib.metadata.version('torch') == '2.11.0+cu130', 'Wrong Colab image: stop.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + [f'{k}=={v}' for k,v in contract['dependencies'].items() if k != 'torch'], check=True)
from t4_smoke import train_smoke as ts
assert not ts.stack_mismatches(ts.versions()), 'Stack mismatch: restart runtime if imports are stale.'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert 'T4' in torch.cuda.get_device_name(0) and torch.cuda.get_device_capability(0) == (7, 5)
from google.colab import drive
# Persistent journals and all attempts stay here across runtime disconnects.
drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/Hinglish_Phase3')
STORE.mkdir(parents=True, exist_ok=True)
RUN = STORE / 'targets_v1'
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass  # Ungated model: anonymous download is allowed.


In [ ]:
# Run generation in a subprocess, freeing GPU memory before CPU selection.
# Set True ONLY after confirming no other evaluator is active for this run.
RECOVER_STALE_LOCK = False
common = ['--root', str(PACKAGE), '--run-dir', str(RUN)]
script = str(PACKAGE / 't4_smoke/phase3_targets.py')
subprocess.run([sys.executable, script, 'generate'] + common + (['--recover-stale-lock'] if RECOVER_STALE_LOCK else []), check=True)
subprocess.run([sys.executable, script, 'select'] + common, check=True)
from google.colab import files
files.download(str(STORE / 'Phase3_Targets_results.zip'))
